In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
texts = [
    "i love this movie",
    "this film is amazing",
    "i enjoyed the story",
    "i hate this movie",
    "this film is terrible",
    "the story was boring"
]

labels = [1, 1, 1, 0, 0, 0]

In [3]:
vocab = {"<PAD>":0}

for sentence in texts:
    for word in sentence.split():
        if word not in vocab:
            vocab[word] = len(vocab)

vocab

{'<PAD>': 0,
 'i': 1,
 'love': 2,
 'this': 3,
 'movie': 4,
 'film': 5,
 'is': 6,
 'amazing': 7,
 'enjoyed': 8,
 'the': 9,
 'story': 10,
 'hate': 11,
 'terrible': 12,
 'was': 13,
 'boring': 14}

In [4]:
max_len = 5
encoded = []

for sentence in texts:
    nums = [vocab[word] for word in sentence.split()]
    # Padding
    while len(nums) < max_len:
        nums.append(0)

    encoded.append(nums)

X = torch.tensor(encoded)
y = torch.tensor(labels)

In [5]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

loader = DataLoader(TextDataset(X, y), batch_size=2, shuffle=True)

In [6]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        output, hidden = self.rnn(x)
        out = self.fc(hidden.squeeze(0))
        return out

In [7]:
model = SentimentRNN(
    vocab_size=len(vocab),
    embed_dim=8,
    hidden_dim=16
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [8]:
for epoch in range(10):
    for inputs, labels in loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.9720
Epoch 2, Loss: 0.7896
Epoch 3, Loss: 0.6838
Epoch 4, Loss: 0.5874
Epoch 5, Loss: 0.7268
Epoch 6, Loss: 0.7000
Epoch 7, Loss: 0.5352
Epoch 8, Loss: 0.4936
Epoch 9, Loss: 0.3032
Epoch 10, Loss: 0.3836


In [9]:
sentence = "i love this film"
nums = [vocab.get(word, 0) for word in sentence.split()]
while len(nums) < max_len:
    nums.append(0)

test = torch.tensor([nums])
model.eval()

with torch.no_grad():
    output = model(test)
    prediction = torch.argmax(output, dim=1)

print("Positive" if prediction.item() == 1 else "Negative")

Positive
